# 0419 · Archivo único de particiones

**Para qué.** Iván pidió (25-10) un solo archivo con las fechas y el número de guías de cada partición: desarrollo, validación de origen móvil (abril, mayo y junio) y prueba final (julio y agosto). Este cuaderno lo arma desde lo que dejó 0411, con las mismas reglas de 0411 (pliegues) y 0416 (prueba final), y comprueba que coincide con los conteos que esos cuadernos imprimieron.

**No entrena nada ni toca la prueba final.** Solo cuenta filas. Tarda segundos.

**Necesita:** `MODELADO/base_modelado.parquet` y `MODELADO/configuracion.json` (0411). **Escribe** `V3/P10_particiones.csv` y `V3/P10_particiones_por_mes.csv`, solo conteos.

**Reglas** (las de `pliegues_origen_movil` y de la celda 7.2 de 0416):
- Entrenamiento de un bloque que empieza en `ini`: T0 < `ini` y etiqueta conocida antes de `ini` (madurez), con objetivo no nulo.
- Validación del bloque: T0 en [`ini`, inicio del bloque siguiente), con objetivo no nulo.
- Prueba final: entrenamiento con T0 y etiqueta antes de `FIN_DESARROLLO` (2026-07-01); prueba con T0 en [2026-07-01, 2026-09-01).

**Control:** si algún conteo no coincide con el impreso en 0411/0416, el cuaderno se detiene. Eso indicaría que la preparación de 0411 cambió después de la prueba final.

In [ ]:
import json
import pandas as pd

PATH_DATABRICKS  = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"
try:
    display
except NameError:
    display = print

# Conteos impresos por 0411 (celdas 19 y 26) y 0416 (celdas 10 y 41) en la corrida del 24/25-10-2026
ESPERADO = {
    ("ambas", "base_modelado", "todas"): 1588872,
    ("clf", "validacion_abril", "entrenamiento"): 585349, ("clf", "validacion_abril", "validacion"): 196990,
    ("clf", "validacion_mayo", "entrenamiento"): 782396,  ("clf", "validacion_mayo", "validacion"): 205304,
    ("clf", "validacion_junio", "entrenamiento"): 988223, ("clf", "validacion_junio", "validacion"): 185663,
    ("clf", "validacion_total", "validacion"): 587957,
    ("clf", "prueba_final", "entrenamiento"): 1173587,    ("clf", "prueba_final", "prueba"): 414913,
    ("reg", "validacion_abril", "entrenamiento"): 584879, ("reg", "validacion_abril", "validacion"): 196989,
    ("reg", "validacion_mayo", "entrenamiento"): 781835,  ("reg", "validacion_mayo", "validacion"): 205301,
    ("reg", "validacion_junio", "entrenamiento"): 986810, ("reg", "validacion_junio", "validacion"): 185661,
    ("reg", "validacion_total", "validacion"): 587951,
    ("reg", "prueba_final", "entrenamiento"): 1172874,    ("reg", "prueba_final", "prueba"): 414903,
}

In [ ]:
# 1. Base y configuración de 0411
CFG = json.load(open(f"{CARPETA_MODELADO}configuracion.json"))
TAREAS = [tuple(t) for t in CFG["TAREAS"]]
INICIOS = [pd.Timestamp(x) for x in CFG["INICIOS_VALIDACION"]]
FIN_DES, FIN_PRUEBA = pd.Timestamp(CFG["FIN_DESARROLLO"]), pd.Timestamp(CFG["FIN_PRUEBA"])
cols = ["t0"] + [c for t in TAREAS for c in t[1:]]
df = pd.read_parquet(f"{CARPETA_MODELADO}base_modelado.parquet", columns=list(dict.fromkeys(cols)))
print(f"{len(df):,} guías · T0 de {df.t0.min()} a {df.t0.max()} · bloques {[i.date() for i in INICIOS]}")

In [ ]:
# 2. Conteos por partición
MES = {4: "abril", 5: "mayo", 6: "junio"}
t_min = df.t0.min().normalize()
filas = [dict(tarea="ambas", particion="base_modelado", rol="todas", t0_desde=t_min.date(), t0_hasta_excl=FIN_PRUEBA.date(),
              condicion=f"guías con margen en T0 y ambas etiquetas (último T0 observado {df.t0.max():%Y-%m-%d %H:%M})",
              guias=len(df))]
limites = INICIOS + [FIN_DES]
for tarea, y, t_et in TAREAS:
    ok = df[y].notna()
    for ini, fin in zip(limites[:-1], limites[1:]):
        nombre = f"validacion_{MES.get(ini.month, ini.date())}"
        filas.append(dict(tarea=tarea, particion=nombre, rol="entrenamiento", t0_desde=t_min.date(), t0_hasta_excl=ini.date(),
                          condicion=f"T0 y etiqueta conocida antes del {ini.date()}",
                          guias=int(((df.t0 < ini) & (df[t_et] < ini) & ok).sum())))
        filas.append(dict(tarea=tarea, particion=nombre, rol="validacion", t0_desde=ini.date(), t0_hasta_excl=fin.date(),
                          condicion=f"T0 en [{ini.date()}, {fin.date()})", guias=int(((df.t0 >= ini) & (df.t0 < fin) & ok).sum())))
    filas.append(dict(tarea=tarea, particion="validacion_total", rol="validacion", t0_desde=INICIOS[0].date(), t0_hasta_excl=FIN_DES.date(),
                      condicion="suma de los tres bloques", guias=int(((df.t0 >= INICIOS[0]) & (df.t0 < FIN_DES) & ok).sum())))
    filas.append(dict(tarea=tarea, particion="prueba_final", rol="entrenamiento", t0_desde=t_min.date(), t0_hasta_excl=FIN_DES.date(),
                      condicion=f"desarrollo: T0 y etiqueta conocida antes del {FIN_DES.date()}",
                      guias=int(((df.t0 < FIN_DES) & (df[t_et] < FIN_DES) & ok).sum())))
    filas.append(dict(tarea=tarea, particion="prueba_final", rol="prueba", t0_desde=FIN_DES.date(), t0_hasta_excl=FIN_PRUEBA.date(),
                      condicion="T0 en julio y agosto; corrida una sola vez el 24-10-2026 (hora de Bogotá)",
                      guias=int(((df.t0 >= FIN_DES) & (df.t0 < FIN_PRUEBA) & ok).sum())))
    # Informativo: guías del desarrollo que no entran al entrenamiento final porque su etiqueta llegó después del corte
    filas.append(dict(tarea=tarea, particion="desarrollo", rol="sin_madurar_al_corte", t0_desde=t_min.date(), t0_hasta_excl=FIN_DES.date(),
                      condicion=f"T0 antes del {FIN_DES.date()} pero etiqueta conocida después",
                      guias=int(((df.t0 < FIN_DES) & ~(df[t_et] < FIN_DES) & ok).sum())))
P = pd.DataFrame(filas)
display(P)

In [ ]:
# 3. Control contra los conteos impresos
P["esperado"] = [ESPERADO.get((r.tarea, r.particion, r.rol)) for r in P.itertuples()]
dif = P[P.esperado.notna() & (P.guias != P.esperado)]
assert dif.empty, f"Conteos distintos a los de 0411/0416:\n{dif}"
print(f"Control OK: {int(P.esperado.notna().sum())} conteos coinciden con 0411/0416")

In [ ]:
# 4. Guías por mes de T0 (con ambas etiquetas, sin filtro de madurez)
pm = (df.assign(mes=df.t0.dt.to_period("M").astype(str))
        .groupby("mes").agg(guias=("t0", "size"), con_etiqueta_clf=(TAREAS[0][1], "count"), con_etiqueta_reg=(TAREAS[1][1], "count"))
        .reset_index())
pm["particion"] = pm.mes.map(lambda m: "prueba_final" if m >= f"{FIN_DES:%Y-%m}" else
                             ("validacion" if m >= f"{INICIOS[0]:%Y-%m}" else "solo_entrenamiento"))
display(pm)

In [ ]:
# 5. Guardar (con pandas: los COPY de DuckDB al Volume fallan en serverless)
P.drop(columns="esperado").to_csv(f"{CARPETA_SALIDAS}P10_particiones.csv", index=False)
pm.to_csv(f"{CARPETA_SALIDAS}P10_particiones_por_mes.csv", index=False)
print("Guardado en", CARPETA_SALIDAS, "· P10_particiones.csv y P10_particiones_por_mes.csv")